# Demo 06: Grounded Hotel Retrieval and Safe Reservations

You run one predictable, graph-enriched hotel retrieval path and one protected reservation command against **your own Neo4j Aura instance** and **Amazon Bedrock**, entirely on your own laptop. This notebook creates no AWS resources.

The full local story: grounded hybrid retrieval, honest abstention when the evidence cannot answer, enforcement of the Neo4j maximum-guests rule, and one idempotent reservation write.

## Who owns what

| Neo4j owns | AWS owns |
|---|---|
| Connected hotel knowledge: hotels, amenities, ratings, policies | Amazon Bedrock reasons over the retrieved evidence |
| The vector index `hotel_chunk_embeddings` and full-text index `hotel_chunk_fulltext` | Amazon Nova 2 creates the query embedding |
| The reviewed Cypher traversal that enriches a matched chunk with its hotel |  |
| The maximum-guests rule and the idempotent `ReservationRequest` write |  |

The retriever is one fixed `HybridCypherRetriever`: explicit `NAIVE` fusion, `top_k=5`, one reviewed traversal. It accepts only a `query`. There is no ranker, alpha, or retriever-mode selector here; those comparisons live in Demo 01b.

In [ ]:
import json
import os
import uuid
from datetime import date, timedelta

import boto3
from dotenv import load_dotenv

from contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from graph_setup import (
    HERO_NAME,
    HERO_SOURCE,
    apply_demo6_graph,
    load_manifest,
    readiness_problems,
)
from hybrid_retrieval import (
    GROUNDING_INSTRUCTIONS,
    Neo4jConfig,
    search_hotel_knowledge,
)
from reservation_command import create_reservation_request
from neo4j import GraphDatabase

load_dotenv()

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-4-6")
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured (set NEO4J_URI/USERNAME/PASSWORD/DATABASE); live cells will be skipped.")
if not BEDROCK_READY:
    print("AWS credentials are not configured; live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to run the hero question.")

## 1. Confirm your Aura connection and prepared indexes

The cell below prepares only the Demo 06 graph-owned data (fixture hotel IDs, uniqueness constraints, and the maximum-guests rule) idempotently, then verifies that both retrieval indexes are online and the hero hotel is present. It never modifies canonical hotel facts.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        problems = apply_demo6_graph(driver, config.database, manifest)
        if not problems:
            problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

## 2. The hero question

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

The exact hotel name benefits from the full-text arm, the request for amenities and a rating benefits from semantic vector matching, and the reviewed traversal adds the connected hotel, its amenities, its rating, and the stable `hotel_id`. This is top-k grounded retrieval, not an exhaustive database listing.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping hero question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']}  (hotel_id={top['hotel_id']})")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Exact matched terms: {', '.join(top['exact_terms']) or 'none'}")
    print(f"Amenities ({len(top['amenities'])}): {', '.join(top['amenities'])}")
    print("\nChunk evidence:")
    print(top["chunk_evidence"][:600])

### What just happened

- **Vector matching** handled the paraphrased request for "amenities and guest rating".
- **Full-text matching** locked onto the exact hotel name and location terms.
- The **reviewed Cypher traversal** followed the matched chunk to its hotel and returned up to 12 connected amenities, the guest rating, and the opaque `hotel_id`. That `hotel_id`, never the display name, is the identity the reservation command accepts later in this notebook.

We do not tune the ranker, alpha, or weights here. The fusion behavior and `top_k` are fixed so the result is deterministic for the workshop.

## 3. A question the evidence cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph holds hotel knowledge, not live inventory. A grounded agent must **abstain** rather than invent availability. Below we wrap the same one retrieval tool in a small local agent with the workshop's grounding instructions, ask the grounded hero question, then ask the unanswerable one.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping grounded agent: retrieval is not configured.")
else:
    from strands import Agent, tool
    from strands.models import BedrockModel

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Search grounded hotel evidence and return bounded JSON facts."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    grounded_agent = Agent(
        model=BedrockModel(
            model_id=MODEL_ID, region_name=AWS_REGION, temperature=0
        ),
        tools=[search_hotel_knowledge_tool],
        system_prompt=(
            "You are a grounded hotel-information assistant. Call "
            "search_hotel_knowledge_tool before answering any hotel question.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    print(grounded_agent(HERO_QUESTION))

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping availability question: retrieval is not configured.")
else:
    print(grounded_agent(AVAILABILITY_QUESTION))

## 4. A 15-guest request is rejected with no write

The Neo4j maximum-guests rule caps a reservation request at 10 guests. The command enforces the rule inside the same boundary as the write, so an over-limit request is rejected and nothing is written to the graph.

These cells need only your Aura connection (`NEO4J_READY`); they do not call Bedrock. The hero `hotel_id` comes from the local fixture manifest, so the write path never depends on a live retrieval result. The dates are computed from today so the example never expires.

In [ ]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    REQUEST_ID = str(uuid.uuid4())
    print(f"Hero hotel_id (from fixture manifest): {hero_id}")
    print(f"Caller-created request_id (reused on retries): {REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}\n")

    over_limit_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        rejected = create_reservation_request(
            over_limit_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print(json.dumps(rejected, indent=2))

## 5. A valid request is recorded, and safe to retry

Reusing the same `request_id`, we submit a request within the 10-guest limit. The command creates one `ReservationRequest` linked to the hero hotel by a `FOR_HOTEL` relationship. Re-delivering the identical request returns the existing record with `duplicate=true` and creates no second node, so a retried or replayed reservation is safe.

In [ ]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        accepted = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
        replay = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

## 6. Inspect the reservation in your graph

Anchored on the stable `request_id`, confirm exactly one accepted request linked to exactly one hotel.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        with driver.session(database=config.database) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## Where this goes next

- **Demo 01b** compares vector, hybrid, Vector-Cypher, and Text2Cypher retrieval patterns. This notebook deliberately fixes one pattern instead of comparing them.
- **Deploying this as a managed service** has its live source in [`deployment-tools/`](deployment-tools/) and a reference walkthrough in [`advanced-deployment/`](advanced-deployment/): the same retrieval contract and reservation command run inside an Amazon Bedrock AgentCore Runtime, fronted by a Gateway and an AWS Lambda. That path, the Runtime, Gateway, Lambda, Dockerfile, and the Secrets Manager and IAM boundary, is retained for reference and is not run in this workshop pass.

Everything above ran on your own Aura instance and Amazon Bedrock, and created no AWS resources.